In [ ]:
# Cell 1: describe all 240 programs and rank their adjusted age-related tail changes.
# Run this cell, inspect the ranked plot, then choose K at the start of cell 2.
# Screen program activities and interpret candidates using the saved fit, without refitting it.
from pathlib import Path
import hashlib
import importlib.metadata
import json
import platform
import time

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import linalg, sparse
from threadpoolctl import threadpool_limits
from tqdm.auto import tqdm

THREAD_LIMIT = threadpool_limits(limits=1)   # limit this notebook's numerical libraries, not other programs
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'margin_dictionary_learning.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Run this notebook from inside the thesis repository.')
RESULTS = ROOT / 'senescence_results'
FIT = RESULTS / 'atlas_fit.h5'
REF = RESULTS / 'references'
ANALYSIS_DIR = RESULTS / 'analysis'
OUT = ANALYSIS_DIR / 'screen'
OUT.mkdir(parents=True, exist_ok=True)
MIN_CELLS = 50
TAIL_FRACTION = 0.15                         # q85: mean of the highest 15%, including zeros
IDENTITY_THRESHOLD = 0.50
LABEL_COLORS = {'identity': '#C47A44', 'activity': '#2678AC'}
started = time.perf_counter()
plt.rcParams.update({'font.family': 'STIXGeneral', 'mathtext.fontset': 'stix',
                     'font.size': 10, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.edgecolor': '0.4',
                     'axes.linewidth': 0.6, 'pdf.fonttype': 42})


def save_figure(fig, name):
    fig.savefig(OUT / f'{name}.pdf', bbox_inches='tight', pad_inches=0.08)
    fig.savefig(OUT / f'{name}.png', dpi=160, bbox_inches='tight', pad_inches=0.08)
    plt.show()
    plt.close(fig)


def upper_tail_mean(values, fraction=TAIL_FRACTION):
    """Use exactly fraction*n cells, with fractional weight at the boundary."""
    n = len(values)
    mass = fraction * n
    whole = int(np.floor(mass))
    boundary = n - whole - 1
    ordered = np.partition(values, boundary, axis=0)
    return (ordered[boundary + 1:].sum(axis=0, dtype=np.float64)
            + (mass - whole) * ordered[boundary]) / mass


def rank_scores(scores):
    """Descending score; exact ties use the original one-based program ID."""
    valid = np.flatnonzero(np.isfinite(scores))
    order = valid[np.lexsort((program_ids[valid], -scores[valid]))]
    ranks = np.full(len(scores), np.nan)
    ranks[order] = np.arange(1, len(order) + 1)
    return ranks


# Read only the fitted activities, dictionary and metadata; not the 14 GB expression file.
fields = ['donor_label', 'donor_age_category', 'donor_sex', 'donor_genotype',
          'region_of_interest_label', 'enrichment_population', 'library_method', 'class_name']
with FIT.open('rb') as stream:
    fit_hash = hashlib.file_digest(stream, 'sha256').hexdigest()
with h5py.File(FIT, 'r') as handle:
    if not handle.attrs['complete'] or handle.attrs['format'] != 'senescence-fit-v1':
        raise ValueError('The fitted atlas is incomplete or has an unexpected format.')
    fit_run_id = handle.attrs['run_id']
    obs = pd.DataFrame({key: pd.Categorical.from_codes(handle[f'obs/{key}/codes'][:],
                         handle[f'obs/{key}/categories'].asstr()[:]) for key in fields})
    D = np.asarray(handle['S'], dtype=np.float64)       # programs x genes; signed weights
    gene_ids = handle['var/gene_ids'].asstr()[:]
    gene_symbols = handle['var/gene_symbols'].asstr()[:]
    activities = sparse.csr_matrix((handle['A/data'][:], handle['A/indices'][:],
                                    handle['A/indptr'][:]), shape=tuple(handle['A'].attrs['shape']))
if D.shape != (240, 23038) or activities.shape != (len(obs), 240) or len(gene_ids) != D.shape[1]:
    raise ValueError('Unexpected fitted axes; this notebook implements the frozen 240-program plan.')
if not np.isfinite(D).all() or not np.isfinite(activities.data).all() or (activities.data < 0).any():
    raise ValueError('Dictionary/activities must be finite; activities must be nonnegative.')
program_ids = np.arange(1, D.shape[0] + 1)              # program 1 is saved dictionary row 0
for field in ['donor_age_category', 'donor_sex', 'donor_genotype']:
    if (obs.groupby('donor_label', observed=True)[field].nunique() != 1).any():
        raise ValueError(f'{field} is not constant within each mouse.')

# Prefilter once. A measurement pools libraries from one mouse, region, sort and class.
cohort = []
def record_cohort(stage, frame):
    ages = frame[['donor_label', 'donor_age_category']].drop_duplicates()
    cohort.append({'stage': stage, 'cells': len(frame),
                   'young_mice': int((ages.donor_age_category == 'adult').sum()),
                   'aged_mice': int((ages.donor_age_category == 'aged').sum()),
                   'classes': frame.class_name.nunique()})

record_cohort('Complete atlas', obs)
obs = obs.loc[(obs.library_method != '10xV3.1') & (obs.donor_genotype != 'wt/wt')].copy()
record_cohort('Exclude 10xV3.1 and wt/wt', obs)
obs = obs.loc[obs.class_name != '08 CNU-MGE GABA'].copy()
record_cohort('Exclude 08 CNU-MGE GABA', obs)
group_keys = ['donor_label', 'region_of_interest_label', 'enrichment_population', 'class_name']
sizes = obs.groupby(group_keys, observed=True).donor_label.transform('size')
obs = obs.loc[sizes >= MIN_CELLS].copy()
for field in fields:
    obs[field] = obs[field].cat.remove_unused_categories()
record_cohort('At least 50 cells per measurement', obs)
grouped = obs.groupby(group_keys, observed=True, sort=True)
group_code = grouped.ngroup().to_numpy()
groups = grouped.agg(n_cells=('donor_label', 'size'),
                     donor_age_category=('donor_age_category', 'first'),
                     donor_sex=('donor_sex', 'first'),
                     donor_genotype=('donor_genotype', 'first')).reset_index()
groups['aged'] = (groups.donor_age_category == 'aged').astype(int)
groups.index.name = 'measurement_id'
# Keep each measurement contiguous: only one small dense measurement is needed at a time.
order = np.argsort(group_code, kind='stable')
obs = obs.iloc[order]
group_code = group_code[order]
activities = activities[obs.index.to_numpy()].tocsr()
source_cell_rows = obs.index.to_numpy()
classes = sorted(groups.class_name.unique())
class_code = pd.Categorical(groups.class_name, categories=classes).codes
mouse_classes = groups[['donor_label', 'class_name', 'donor_age_category']].drop_duplicates()
class_counts = pd.crosstab(mouse_classes.class_name, mouse_classes.donor_age_category).reindex(classes)
expected = (1061512, 1288, 95, 683, 24, 57, 38)
actual = (len(obs), len(groups), groups.donor_label.nunique(), len(mouse_classes), len(classes),
          cohort[-1]['young_mice'], cohort[-1]['aged_mice'])
if actual != expected:
    raise ValueError(f'Cohort differs from the agreed frozen data: {actual} != {expected}')
display(pd.DataFrame(cohort), class_counts.rename(columns={'adult': 'young_mice', 'aged': 'aged_mice'}))

# Step 1 / 2a: retain zeros in means and in the exact upper-tail calculation.
M = np.empty((len(groups), len(program_ids)), dtype=np.float64)
U = np.empty_like(M)
audit_tails = {fraction: np.empty_like(M) for fraction in (0.20, 0.10)}
offsets = np.r_[0, np.cumsum(groups.n_cells.to_numpy())]
for r in tqdm(range(len(groups)), desc='Measurement means and q85 tails', mininterval=5):
    values = activities[offsets[r]:offsets[r + 1]].toarray().astype(np.float64)
    M[r] = values.mean(axis=0)
    U[r] = upper_tail_mean(values)
    for fraction in audit_tails:
        audit_tails[fraction][r] = upper_tail_mean(values, fraction)
del values
R = groups.groupby(['donor_label', 'class_name'], observed=True).donor_label.transform('size').to_numpy()
B = np.array([class_counts.loc[c, g] for c, g in zip(groups.class_name, groups.donor_age_category)])
reference_weight = 1.0 / (2 * B * R)         # equal ages, mice within age, measurements within mouse
regression_weight = 1.0 / R                 # each mouse-class pair totals 1; no age equalization
reference = np.eye(len(classes))[class_code].T * reference_weight
mu = (reference @ M).T
U_ref = (reference @ U).T
total_mean = mu.sum(axis=1)
w = np.divide(mu, total_mean[:, None], out=np.full_like(mu, np.nan), where=total_mean[:, None] > 0)
T = np.sum(w * U_ref, axis=1)                # typical tail level: one denominator per program
active = total_mean > 0
if not np.allclose(w[active].sum(axis=1), 1) or not np.all(np.isfinite(T[active]) & (T[active] > 0)):
    raise ValueError('Invalid class weights or reference tail denominator.')
labels = np.where(w.max(axis=1) >= IDENTITY_THRESHOLD, 'identity', 'activity').astype(object)
labels[~active] = None                      # inactive programs are retained, but not ranked
if (~active).any():
    print('Entirely inactive, unrankable programs:', program_ids[~active].tolist())

# Diagnostic justification for q85; q80/q90 are not alternative candidate screens.
audit_rows = []
diagnostic_weight = reference_weight[:, None] * w[:, class_code].T * (M > 0)
diagnostic_weight /= diagnostic_weight.sum(axis=0)
for fraction, tails in [(0.20, audit_tails[0.20]), (TAIL_FRACTION, U), (0.10, audit_tails[0.10])]:
    captured = np.divide(fraction * tails, M, out=np.zeros_like(M), where=M > 0)
    audit_rows.append({'quantile': int(round(100 * (1 - fraction))),
                      'median_activity_share': np.nanmedian((diagnostic_weight * captured).sum(axis=0)),
                      'median_nearly_all_activity': np.nanmedian((diagnostic_weight * (captured >= 0.95)).sum(axis=0))})
pd.DataFrame(audit_rows).to_csv(OUT / 'tail_diagnostics.csv', index=False)
del audit_tails

# Step 2b: 24 class baselines + 24 age contrasts + shared additive adjustments.
class_design = np.eye(len(classes))[class_code]
X = np.column_stack([class_design, class_design * groups.aged.to_numpy()[:, None]])
design_columns = [f'class[{c}]' for c in classes] + [f'aged[{c}]' for c in classes]
age_columns = np.arange(len(classes), 2 * len(classes))
reference_levels = {}
for field in ['donor_sex', 'region_of_interest_label', 'enrichment_population', 'donor_genotype']:
    levels = sorted(groups[field].unique())
    reference_levels[field] = levels[0]
    adjustment = pd.get_dummies(pd.Categorical(groups[field], categories=levels), drop_first=True, dtype=float)
    X = np.column_stack([X, adjustment.to_numpy()])
    design_columns.extend(f'{field}[{level}]' for level in levels[1:])
if X.shape != (1288, 70):
    raise ValueError(f'Unexpected regression design: {X.shape}')
weighted_X = X * np.sqrt(regression_weight)[:, None]
weighted_Y = np.column_stack([M, U]) * np.sqrt(regression_weight)[:, None]


def fit_coefficients(keep=None):
    """A shared QR solves all 240 M and 240 U regressions; no age p-values are used."""
    keep = slice(None) if keep is None else keep
    with threadpool_limits(limits=1):
        Q, triangular = linalg.qr(weighted_X[keep], mode='economic')
        if np.linalg.matrix_rank(triangular) != X.shape[1]:
            raise ValueError('Regression design lost rank; do not report unidentified age effects.')
        return linalg.solve_triangular(triangular, Q.T @ weighted_Y[keep])


def fit_age_effects(keep=None):
    coefficients = fit_coefficients(keep)
    return (coefficients[age_columns, :len(program_ids)].T,
            coefficients[age_columns, len(program_ids):].T)


coefficients = fit_coefficients()
delta_m = coefficients[age_columns, :len(program_ids)].T
delta_u = coefficients[age_columns, len(program_ids):].T
s = delta_u - np.maximum(delta_m, 0)         # safe class score: tail gain beyond a positive mean shift
raw_scores = np.sum(w * s, axis=1)           # negative class contributions remain negative
scores = raw_scores / T                     # dimensionless; cancels multiplicative program scale
all_programs = pd.DataFrame({'program_id': program_ids, 'rank': rank_scores(scores), 'score': scores,
                             'raw_score': raw_scores, 'reference_tail': T, 'label': labels,
                             'dominant_class': np.asarray(classes)[np.argmax(mu, axis=1)],
                             'max_class_weight': w.max(axis=1)}).set_index('program_id')
all_programs.loc[~active, 'dominant_class'] = None
all_programs['rank'] = all_programs['rank'].astype('Int64')
ranked = all_programs.sort_values(['rank'], na_position='last')

# Save the numerical record, including nuisance coefficients and reference-building intermediates.
groups.assign(regression_weight=regression_weight, reference_weight=reference_weight).to_csv(OUT / 'measurements.csv')
pd.DataFrame(cohort).to_csv(OUT / 'cohort.csv', index=False)
class_counts.to_csv(OUT / 'class_mouse_counts.csv')
ranked.to_csv(OUT / 'program_scores.csv')
pd.DataFrame(X, columns=design_columns).to_csv(OUT / 'regression_design.csv', index_label='measurement_id')
np.savez_compressed(OUT / 'numerical_results.npz', M=M, U=U, mu=mu, U_ref=U_ref, w=w, T=T,
                    coefficients=coefficients, source_cell_rows=source_cell_rows, group_code=group_code,
                    classes=np.asarray(classes), program_ids=program_ids, design_columns=np.asarray(design_columns))
class_results = pd.DataFrame({'program_id': np.repeat(program_ids, len(classes)),
                              'class_name': np.tile(classes, len(program_ids)),
                              'mean_reference': mu.ravel(), 'tail_reference': U_ref.ravel(),
                              'usage_weight': w.ravel(), 'age_mean': delta_m.ravel(), 'age_tail': delta_u.ravel(),
                              'safe_class_score': s.ravel(), 'score_contribution': (w * s / T[:, None]).ravel()})
class_results.to_csv(OUT / 'class_results.csv', index=False)
for outcome, values in [('mean', M), ('tail', U)]:
    mouse_reference = pd.concat([groups[['donor_label', 'class_name', 'donor_age_category']],
                                 pd.DataFrame(values, columns=program_ids)], axis=1).groupby(
        ['donor_label', 'class_name', 'donor_age_category'], observed=True).mean()
    mouse_reference.to_csv(OUT / f'mouse_class_{outcome}.csv')
    mouse_reference.groupby(['class_name', 'donor_age_category'], observed=True).mean().to_csv(OUT / f'age_class_{outcome}.csv')
software = {name: importlib.metadata.version(name) for name in
            ['numpy', 'scipy', 'pandas', 'h5py', 'matplotlib', 'threadpoolctl', 'gseapy']}
provenance = {'fit_sha256': fit_hash, 'fit_run_id': fit_run_id, 'program_ids': 'one-based dictionary row',
              'python': platform.python_version(), 'software': software, 'tail_fraction': TAIL_FRACTION,
              'minimum_cells': MIN_CELLS, 'identity_threshold': IDENTITY_THRESHOLD,
              'reference': 'equal measurements within mouse-class; equal mice within age-class; equal ages',
              'regression_weights': '1 / measurements per mouse-class', 'reference_levels': reference_levels,
              'cohort': cohort, 'design_rank': int(np.linalg.matrix_rank(X)),
              'inactive_programs': program_ids[~active].tolist()}
(OUT / 'settings.json').write_text(json.dumps(provenance, indent=2), encoding='utf8')

# Step 1 overview; the separate ranked plot below is the only input for choosing K.
fig, ax = plt.subplots(figsize=(10, 11))
heat = ax.imshow(w, aspect='auto', interpolation='nearest', vmin=0, vmax=1, cmap='viridis')
ax.set_xticks(np.arange(len(classes)), classes, rotation=60, ha='right')
ax.set_yticks(np.arange(0, len(program_ids), 10), program_ids[::10])
ax.set(xlabel='Broad cell class', ylabel='Program ID', title='Age-balanced class usage: all 240 programs')
fig.colorbar(heat, ax=ax, label='Share of reference class means', shrink=0.55)
save_figure(fig, 'class_usage_all_programs')
fig, ax = plt.subplots(figsize=(8, 4))
for label, color in LABEL_COLORS.items():
    subset = ranked[ranked.label == label]
    ax.scatter(subset['rank'], subset.score, s=13, color=color, label=label.capitalize(), zorder=3)
ax.axhline(0, color='0.45', linewidth=0.7)
ax.set(xlabel='Rank among all programs', ylabel='Tail-normalized age score', title='Choose K from this ranking')
ax.legend(frameon=False)
save_figure(fig, 'ranked_program_scores')
display(ranked.head(20))
print(f'Cell 1: {time.perf_counter() - started:.1f} seconds. Outputs: {OUT}')
print('Inspect the full ranked plot, then set K and record your reason in cell 2 before inspecting genes.')


In [ ]:
# Cell 2: after inspecting cell 1, fix the top K and produce all candidate results.
# K is the number of candidates; the fitted dictionary always has 240 programs.
# SenMayo results are written to separate CSVs, never displayed here. Write your
# Reactome interpretation before opening those files; there is no computational gate.
K = 7
K_REASON = 'Selected from the score curve before molecular interpretation: the mean score decline is 0.0666 per rank over ranks 1-7 and 0.00687 over ranks 7-13. Seven is a practical elbow for focused interpretation, not a significance threshold.'
started = time.perf_counter()

positive = all_programs.loc[all_programs.score > 0].sort_values('rank')
if not isinstance(K, int) or isinstance(K, bool) or not 1 <= K <= len(positive):
    raise ValueError(f'Choose an integer K from 1 to {len(positive)} positive-score programs. '
                     'If there are none, this screen has no positive candidates.')
top = positive.head(K).copy()
selected = top.index.to_numpy() - 1          # one-based IDs map to zero-based dictionary rows
selection_hash = hashlib.sha256((json.dumps(provenance, sort_keys=True) + top.to_csv()).encode()
                                + (REF / 'derived_checksums.json').read_bytes()).hexdigest()[:12]
OUT = ANALYSIS_DIR / f'top_{K}_{selection_hash}'
OUT.mkdir(parents=True, exist_ok=True)
selection = {'K_candidates': K, 'program_ids': top.index.tolist(), 'reason': K_REASON,
             'boundary_score': float(top.score.iloc[-1]), 'fit_sha256': fit_hash,
             'selection_fingerprint': selection_hash,
             'selection_rule': 'highest positive normalized scores; exact ties use ascending program ID'}
(OUT / 'candidate_selection.json').write_text(json.dumps(selection, indent=2), encoding='utf8')
top.to_csv(OUT / 'selected_programs.csv')

# Record the chosen boundary on the original ranking before examining genes or omissions.
fig, ax = plt.subplots(figsize=(8, max(4, .20 * K)))
for label, color in LABEL_COLORS.items():
    subset = all_programs[all_programs.label == label]
    ax.scatter(subset['rank'], subset.score, s=13, color=color, label=label.capitalize())
ax.axhline(0, color='0.45', lw=.7)
ax.axvline(K + .5, color='0.6', lw=.8, ls='--')
ax.set(xlabel='Rank among all programs', ylabel='Tail-normalized age score',
       title=f'Fixed top {K}: numbers identify dictionary programs')
ax.legend(frameon=False)
# Separate nearby labels in display coordinates; lines connect them to their actual dots.
fig.canvas.draw()
label_y = ax.transData.transform(top[['rank', 'score']].to_numpy(dtype=float))[:, 1]
for i in range(1, K):
    label_y[i] = min(label_y[i], label_y[i - 1] - 10 * fig.dpi / 72)
for (program, row), height in zip(top.iterrows(), label_y):
    label_height = ax.transData.inverted().transform((0, height))[1]
    ax.annotate(str(program), (row['rank'], row.score), xytext=(K + 7, label_height),
                fontsize=8, va='center', arrowprops={'arrowstyle': '-', 'lw': .5, 'color': '0.4'})
save_figure(fig, 'ranked_scores_with_candidates')

# Leave out whole mice, keeping the full-cohort reference and shortlist fixed.
candidate_colors = [LABEL_COLORS[label] for label in top['label']]
candidate_labels = [f'Program {p} ({label})' for p, label in zip(top.index, top['label'])]
mice = np.sort(groups['donor_label'].unique())
loo_raw = np.full((len(mice), len(program_ids)), np.nan)
loo_score = np.full_like(loo_raw, np.nan)
loo_rank = np.full_like(loo_raw, np.nan)
with threadpool_limits(limits=1):
    for i, mouse in enumerate(mice):
        keep = groups['donor_label'].ne(mouse).to_numpy()
        omission_m, omission_u = fit_age_effects(keep)
        loo_raw[i] = np.sum(w * (omission_u - np.maximum(omission_m, 0)), axis=1)
        np.divide(loo_raw[i], T, out=loo_score[i], where=np.isfinite(T) & (T > 0))
        loo_rank[i] = rank_scores(loo_score[i])
pd.DataFrame({
    'omitted_mouse': np.repeat(mice, len(program_ids)),
    'program_id': np.tile(program_ids, len(mice)),
    'raw_score': loo_raw.ravel(), 'score': loo_score.ravel(), 'rank': loo_rank.ravel(),
}).to_csv(OUT / 'mouse_omission_scores.csv', index=False)
influence = top[['rank', 'label', 'score', 'raw_score', 'reference_tail']].copy()
influence['minimum_score'] = np.min(loo_score[:, selected], axis=0)
influence['maximum_score'] = np.max(loo_score[:, selected], axis=0)
influence['minimum_rank'] = np.min(loo_rank[:, selected], axis=0).astype(int)
influence['maximum_rank'] = np.max(loo_rank[:, selected], axis=0).astype(int)
influence['nonpositive_after_omission'] = np.any(loo_score[:, selected] <= 0, axis=0)
influence['mouse_at_minimum_score'] = mice[np.argmin(loo_score[:, selected], axis=0)]
influence['mouse_at_maximum_score'] = mice[np.argmax(loo_score[:, selected], axis=0)]
influence.to_csv(OUT / 'candidate_mouse_influence.csv', index_label='program_id')
print(f'Completed {len(mice)} whole-mouse omissions; all fits retain the full design.')

# These whiskers describe single-mouse influence, not sampling confidence intervals.
y = np.arange(len(top))
fig, axes = plt.subplots(1, 2, figsize=(9, 1.8 + .43 * len(top)), sharey=True)
for ax, field, lower, upper in zip(
        axes, ['score', 'rank'], ['minimum_score', 'minimum_rank'],
        ['maximum_score', 'maximum_rank']):
    ax.hlines(y, influence[lower], influence[upper], color=candidate_colors, lw=1.7)
    ax.scatter(influence[field], y, c=candidate_colors, s=35, edgecolor='white', lw=.6, zorder=3)
    ax.set_yticks(y, candidate_labels)
    ax.grid(axis='x', color='#E8EBEE', lw=.6)
    ax.set_axisbelow(True)
axes[0].invert_yaxis()
axes[0].axvline(0, color='#929BA3', lw=.8, ls='--')
axes[1].axvline(len(top), color='#929BA3', lw=.8, ls='--')
axes[0].set_xlabel('Tail-normalized age score')
axes[1].set_xlabel('Rank among all programs (1 is best)')
fig.suptitle('Range after leaving out one mouse', fontsize=10)
fig.tight_layout()
save_figure(fig, 'candidate_mouse_influence')

# Class contributions use one denominator per program and sum to its final score.
contribution = w[selected] * s[selected] / T[selected, None]
np.testing.assert_allclose(contribution.sum(axis=1), top['score'].to_numpy(), rtol=1e-10, atol=1e-12)
pd.DataFrame(w[selected], index=top.index, columns=classes).to_csv(
    OUT / 'candidate_class_usage.csv', index_label='program_id')
pd.DataFrame(contribution, index=top.index, columns=classes).to_csv(
    OUT / 'candidate_class_contributions.csv', index_label='program_id')
fig, axes = plt.subplots(1, 2, figsize=(14, 3.0 + .35 * len(top)), sharey=True)
usage_image = axes[0].imshow(w[selected], aspect='auto', cmap='viridis', vmin=0, vmax=1)
limit = np.max(np.abs(contribution))
contribution_image = axes[1].imshow(contribution, aspect='auto', cmap='RdBu_r', vmin=-limit, vmax=limit)
for ax, title in zip(axes, ['Age-balanced class usage', 'Contribution to normalized age score']):
    ax.set_title(title)
    ax.set_xticks(np.arange(len(classes)), classes, rotation=90, fontsize=8)
    ax.set_yticks(y, candidate_labels)
    ax.tick_params(length=0)
fig.colorbar(usage_image, ax=axes[0], shrink=.75, label='Class weight')
fig.colorbar(contribution_image, ax=axes[1], shrink=.75, label='Score contribution (unitless)')
fig.tight_layout()
save_figure(fig, 'candidate_class_usage_and_contributions')

# Within age and class, every mouse has weight one; split it across measurements
# and then cells. Mix classes with the same program-specific w for both ages.
R_group = groups.groupby(['donor_label', 'class_name'], observed=True)['n_cells'].transform('size').to_numpy()
B_group = groups.groupby(['aged', 'class_name'], observed=True)['donor_label'].transform('nunique').to_numpy()
base_weight = 1.0 / (R_group * B_group * groups['n_cells'].to_numpy())
group_class = pd.Categorical(groups['class_name'], categories=classes).codes
group_age = groups['aged'].to_numpy(dtype=int)
assert np.all(group_class >= 0) and set(group_age) == {0, 1}
age_class = group_age * len(classes) + group_class
group_total = np.bincount(age_class, weights=base_weight * groups['n_cells'].to_numpy(), minlength=2 * len(classes))
np.testing.assert_allclose(group_total, 1, rtol=1e-12, atol=1e-12)
candidate_activity = activities[:, selected].tocsc()
candidate_activity.sum_duplicates()
candidate_activity.eliminate_zeros()
histogram_tables, class_histogram_tables, zero_tables = [], [], []
fig, axes = plt.subplots(len(top), 2, figsize=(9, 1.0 + 2.0 * len(top)),
                         squeeze=False, sharex='row', sharey='row')
age_names, age_colors = ['young', 'aged'], ['#2678AC', '#C47A44']
for row, k in enumerate(selected):
    start, end = candidate_activity.indptr[row:row + 2]
    cell_index = candidate_activity.indices[start:end]
    values = candidate_activity.data[start:end]
    assert len(values) and np.all(np.isfinite(values)) and np.all(values > 0)
    edges = np.linspace(0, float(values.max()), 51)
    value_group = group_code[cell_index]
    value_age_class = age_class[value_group]
    bins = np.minimum(np.searchsorted(edges, values, side='right') - 1, 49)
    # Implicit sparse zeros must contribute to the first bin, not disappear.
    nonzero_mass = np.bincount(value_age_class, weights=base_weight[value_group],
                               minlength=2 * len(classes)).reshape(2, len(classes))
    assert np.all(nonzero_mass <= 1 + 1e-10)
    class_zero = np.clip(1 - nonzero_mass, 0, 1)
    class_hist = np.bincount(value_age_class * 50 + bins, weights=base_weight[value_group],
                             minlength=2 * len(classes) * 50).reshape(2, len(classes), 50)
    class_hist[:, :, 0] += class_zero
    np.testing.assert_allclose(class_hist.sum(axis=2), 1, rtol=1e-10, atol=1e-12)
    mixed_hist = np.einsum('c,acj->aj', w[k], class_hist)
    mixed_zero = class_zero @ w[k]
    np.testing.assert_allclose(mixed_hist.sum(axis=1), 1, rtol=1e-10, atol=1e-12)
    for age in (0, 1):
        ax = axes[row, age]
        ax.stairs(mixed_hist[age], edges, fill=True, color=age_colors[age], alpha=.8, lw=.6)
        ax.set(xlim=(edges[0], edges[-1]), ylim=(0, 1.08 * mixed_hist.max()),
               title=f'Program {program_ids[k]}: {age_names[age]}', xlabel='Fitted program activity')
        ax.text(.98, .90, f'Zero activity: {mixed_zero[age]:.1%}', transform=ax.transAxes,
                ha='right', va='top', fontsize=9)
        if age == 0:
            ax.set_ylabel('Weighted cell fraction per bin')
        histogram_tables.append(pd.DataFrame({
            'program_id': program_ids[k], 'age': age_names[age], 'bin_index': np.arange(50),
            'bin_left': edges[:-1], 'bin_right': edges[1:], 'bin_fraction': mixed_hist[age],
        }))
        class_histogram_tables.append(pd.DataFrame({
            'program_id': program_ids[k], 'age': age_names[age],
            'class_name': np.repeat(classes, 50), 'bin_index': np.tile(np.arange(50), len(classes)),
            'bin_left': np.tile(edges[:-1], len(classes)), 'bin_right': np.tile(edges[1:], len(classes)),
            'class_bin_fraction': class_hist[age].ravel(), 'class_weight': np.repeat(w[k], 50),
            'contribution_to_mixture': (w[k, :, None] * class_hist[age]).ravel(),
        }))
        zero_tables.append(pd.DataFrame({
            'program_id': program_ids[k], 'age': age_names[age], 'class_name': list(classes) + ['mixture'],
            'zero_fraction': np.r_[class_zero[age], mixed_zero[age]],
        }))
fig.tight_layout()
save_figure(fig, 'candidate_age_distributions')
pd.concat(histogram_tables, ignore_index=True).to_csv(OUT / 'candidate_activity_histograms.csv', index=False)
pd.concat(class_histogram_tables, ignore_index=True).to_csv(OUT / 'candidate_class_histograms.csv', index=False)
pd.concat(zero_tables, ignore_index=True).to_csv(OUT / 'candidate_zero_fractions.csv', index=False)
del candidate_activity

# Steps 5--6 and Appendix A: molecular evidence for the already selected top K.
# All computations run now. SenMayo outputs are saved separately and never displayed:
# write the functional interpretation from Reactome before opening those CSVs.
import gseapy
from statsmodels.stats.multitest import multipletests


def annotate_top_programs(D, gene_ids, gene_symbols, top, REF, OUT):
    """Save reproducible molecular evidence; leave scientific judgments to the reader."""
    ann = OUT / "molecular_evidence"
    cache = OUT.parent / "enrichment_cache"  # Reuse individual programs when K changes.
    for folder in (ann, cache):
        folder.mkdir(parents=True, exist_ok=True)
    assert importlib.metadata.version("gseapy") == "1.3.1", "Use the pinned GSEApy 1.3.1."
    parameters = dict(method="multilevel", sample_size=101, eps=1e-10,
                      permutation_num=1000, weight=1, seed=2026, min_size=15,
                      max_size=500, threads=1, ascending=None, no_plot=True,
                      outdir=None, verbose=False, organism="mouse")
    # Exact ties use gene-ID order below; GSEApy's generic tie warning does not change it.
    files = ["atlas_gene_identifiers.tsv", "reactome_mouse_ensembl.gmt",
             "senmayo_mouse_ensembl.gmt", "senmayo_mouse_identifier_mapping.tsv",
             "reactome_mouse_pathway_relations.tsv", "reactome_mouse_pathways.tsv"]
    hashes = {name: hashlib.sha256((REF / name).read_bytes()).hexdigest() for name in files}
    frozen = json.loads((REF / "derived_checksums.json").read_text(encoding="utf-8"))
    for name in files:
        assert hashes[name] == frozen[name]["sha256"], f"Frozen reference differs: {name}"
    relations = pd.read_csv(REF / files[4], sep="\t")
    pathway_metadata = pd.read_csv(REF / files[5], sep="\t").set_index("pathway_id")
    branch = {"R-MMU-2559583"}
    while True:
        expanded = branch | set(relations.loc[relations.parent_id.isin(branch), "child_id"])
        if expanded == branch:
            break
        branch = expanded
    mapping = pd.read_csv(REF / files[0], sep="\t", keep_default_na=False)
    assert np.array_equal(mapping.gene_index.to_numpy(), np.arange(D.shape[1]))
    assert np.array_equal(mapping.atlas_gene_id.to_numpy(str), np.asarray(gene_ids, str))
    reason = mapping.mapping_status.astype(str).copy()
    reason.loc[mapping.atlas_gene_id.duplicated(False)] = "duplicate_atlas_identifier"
    one = reason.eq("one_to_one")
    repeated_mgi = mapping.loc[one, "mgi_ids"].duplicated(False)
    reason.loc[repeated_mgi.index[repeated_mgi]] = "duplicate_biological_gene"
    keep = reason.eq("one_to_one").to_numpy()
    universe = mapping.loc[keep].copy()
    universe.to_csv(ann / "gene_universe.csv", index=False)
    mapping.loc[~keep].assign(exclusion_reason=reason[~keep]).to_csv(
        ann / "excluded_gene_identifiers.csv", index=False)
    ids = universe.atlas_gene_id.to_numpy(str)
    id_set = set(ids)
    symbols = universe.mgi_current_symbols.to_numpy(str)
    norms = np.linalg.norm(D, axis=0)
    assert np.isfinite(D).all() and np.isfinite(norms).all() and (norms > 0).all(), (
        "Unexpected zero/nonfinite dictionary column: inspect it; do not change the universe silently.")
    pd.DataFrame({"gene_index": np.arange(D.shape[1]), "gene_id": gene_ids,
                  "atlas_symbol": gene_symbols, "loading_norm_all_240": norms}).to_csv(
        ann / "dictionary_gene_loading_norms.csv", index=False)

    sets, names, coverage = {}, {}, []
    for collection, filename in (("reactome", files[1]), ("senmayo", files[2])):
        sets[collection], names[collection] = {}, {}
        for line in (REF / filename).read_text(encoding="utf-8").splitlines():
            term, description, *members = line.split("\t")
            original, overlap = set(members), sorted(set(members) & id_set)
            eligible = 15 <= len(overlap) <= 500
            coverage.append(dict(collection=collection, pathway_id=term,
                                 pathway_name=description, frozen_set_size=len(original),
                                 universe_overlap=len(overlap), eligible=eligible))
            names[collection][term] = description
            if eligible:
                sets[collection][term] = overlap
    coverage = pd.DataFrame(coverage)
    coverage["cellular_senescence_branch"] = coverage.pathway_id.isin(branch)
    pathway_metadata.assign(cellular_senescence_branch=pathway_metadata.index.isin(branch)).to_csv(
        ann / "reactome_pathway_metadata.csv")
    # Keep even coverage/membership information about SenMayo in separately named files.
    for collection in sets:
        coverage.query("collection == @collection").to_csv(
            ann / f"{collection}_coverage.csv", index=False)
    senmap = pd.read_csv(REF / files[3], sep="\t", keep_default_na=False)
    senmap.assign(in_analysis_universe=senmap.unique_ensembl_id.isin(id_set)).to_csv(
        ann / "senmayo_published_gene_coverage.csv", index=False)
    if not sets["reactome"]:
        raise ValueError("No Reactome pathways pass the fixed 15--500 overlap rule.")

    fingerprint = hashlib.sha256(D.tobytes() + json.dumps(
        {"references": hashes, "parameters": parameters, "gseapy": "1.3.1",
         "universe": ids.tolist(), "protocol": "signed-gene-enrichment-v1"},
        sort_keys=True).encode()).hexdigest()
    rankings, tie_rows = {}, []
    for program in top.index:
        row = int(program) - 1
        raw, specificity = D[row, keep], D[row, keep] / norms[keep]
        frame = pd.DataFrame({"program_id": int(program), "gene_index": np.flatnonzero(keep),
                              "gene_id": ids, "gene_symbol": symbols, "raw_weight": raw,
                              "loading_specificity": specificity,
                              "loading_norm_all_240": norms[keep]})
        for metric, values in (("raw", raw), ("specificity", specificity)):
            order = np.lexsort((ids, -values))
            positions = np.empty(len(order), dtype=int)
            positions[order] = np.arange(1, len(order) + 1)
            frame[f"{metric}_rank"] = positions
            tie_rows.append(dict(program_id=int(program), metric=metric,
                                 tied_values_beyond_first=len(values) - len(np.unique(values)),
                                 tied_gene_fraction=pd.Series(values).duplicated(False).mean()))
        rankings[int(program)] = frame
    gene_evidence = pd.concat(rankings.values(), ignore_index=True)
    gene_evidence.to_csv(ann / "gene_weights_specificity_and_ranks.csv.gz", index=False,
                         compression={"method": "gzip", "compresslevel": 1})
    pd.DataFrame(tie_rows).to_csv(ann / "ranking_ties.csv", index=False)

    def enrich(program, metric, collection):
        path = cache / f"{fingerprint[:20]}_{program:03d}_{metric}_{collection}.csv"
        if path.exists():
            return pd.read_csv(path, keep_default_na=False, na_values=[""], float_precision="round_trip")
        frame = rankings[program].sort_values(f"{metric}_rank")
        column = "raw_weight" if metric == "raw" else "loading_specificity"
        rnk = frame[["gene_id", column]].copy()
        if not sets[collection]:
            # An unassessable signature is not evidence of absence.
            result = coverage.query("collection == @collection").copy()
            result = result[["pathway_id", "pathway_name", "universe_overlap"]]
            result = result.assign(ES=np.nan, NES=np.nan, p_value=np.nan,
                                   leading_edge_ids="", status="not_assessable_coverage")
        else:
            fit = gseapy.prerank(rnk=rnk, gene_sets=sets[collection], **parameters)
            result = fit.res2d.rename(columns={"Term": "pathway_id", "NOM p-val": "p_value",
                                              "Lead_genes": "leading_edge_ids"}).copy()
            cols = ["pathway_id", "ES", "NES", "p_value", "leading_edge_ids"]
            if "log2err" in result:
                cols.append("log2err")
            result = result[cols]
            result["pathway_name"] = result.pathway_id.map(names[collection])
            result["universe_overlap"] = result.pathway_id.map(
                {term: len(members) for term, members in sets[collection].items()})
            for col in ("ES", "NES", "p_value"):
                result[col] = pd.to_numeric(result[col], errors="coerce")
            result["status"] = np.where(np.isfinite(result.p_value), "tested", "not_estimable")
            assert set(result.pathway_id) == set(sets[collection]), "GSEA silently dropped a pathway."
            assert ((result.p_value.dropna() >= 0) & (result.p_value.dropna() <= 1)).all()
        result = result.assign(program_id=program, metric=metric, collection=collection)
        temporary = path.with_suffix(".partial.csv")
        result.to_csv(temporary, index=False)
        temporary.replace(path)
        return result

    results = {}
    # Primary and supplementary Reactome precede the separately saved benchmark.
    for collection in ("reactome", "senmayo"):
        for metric in ("raw", "specificity"):
            print(f"Molecular annotation: {collection}, {metric}, {len(top)} selected programs")
            table = pd.concat([enrich(int(p), metric, collection) for p in top.index], ignore_index=True)
            tested = pd.to_numeric(table.p_value, errors="coerce").notna()
            # Numerical failures must not shrink the prespecified testing family.
            # Use p=1 only inside BH, then leave their reported p/q unavailable.
            table["q_BH"] = multipletests(table.p_value.fillna(1).astype(float), method="fdr_bh")[1]
            table.loc[~tested, "q_BH"] = np.nan
            table["age_score_rank"] = table.program_id.map(top["rank"])
            table["cellular_senescence_branch"] = table.pathway_id.isin(branch)
            table = table.sort_values(["age_score_rank", "q_BH", "pathway_id"], na_position="last")
            table.to_csv(ann / f"{collection}_{metric}_enrichment.csv", index=False)
            if collection == "reactome":
                table[table.cellular_senescence_branch].to_csv(
                    ann / f"reactome_{metric}_cellular_senescence_branch.csv", index=False)
            results[collection, metric] = table
            leading = table[["program_id", "pathway_id", "pathway_name", "NES", "q_BH",
                             "cellular_senescence_branch", "leading_edge_ids"]].copy()
            leading["gene_id"] = leading.leading_edge_ids.fillna("").str.split(";")
            leading = leading.explode("gene_id").drop(columns="leading_edge_ids")
            leading = leading[leading.gene_id.ne("")].merge(
                gene_evidence, on=["program_id", "gene_id"], how="left", validate="many_to_one")
            leading.to_csv(ann / f"{collection}_{metric}_leading_gene_evidence.csv.gz", index=False,
                           compression={"method": "gzip", "compresslevel": 1})

    primary = results["reactome", "raw"]
    alternative = results["reactome", "specificity"]
    comparison = primary.merge(alternative, on=["program_id", "pathway_id"],
                               suffixes=("_raw", "_specificity"), validate="one_to_one")
    comparison["same_NES_direction"] = (np.sign(comparison.NES_raw) == np.sign(comparison.NES_specificity)).astype(
        "boolean").mask(comparison[["NES_raw", "NES_specificity"]].isna().any(axis=1))
    for metric in ("raw", "specificity"):
        q_values = comparison[f"q_BH_{metric}"]
        comparison[f"supported_{metric}"] = q_values.le(.05).astype("boolean").mask(q_values.isna())
    comparison.to_csv(ann / "reactome_appendix_comparison.csv", index=False)
    benchmark = results["senmayo", "raw"].merge(results["senmayo", "specificity"],
        on=["program_id", "pathway_id"], suffixes=("_raw", "_specificity"), validate="one_to_one")
    benchmark.to_csv(ann / "senmayo_appendix_comparison.csv", index=False)
    sen_genes = set(senmap.loc[senmap.mapping_status.eq("one_to_one"), "unique_ensembl_id"])
    gene_evidence.assign(in_senmayo=gene_evidence.gene_id.isin(sen_genes)).to_csv(
        ann / "senmayo_gene_membership_for_later_annotation.csv.gz", index=False,
        compression={"method": "gzip", "compresslevel": 1})

    # This is a working table, not an automatic biological verdict. Preserve manual edits.
    review_path = ann / f"functional_interpretation_top_{len(top)}.csv"
    review = top.copy().rename_axis("program_id").reset_index()
    human_fields = ["theme_1", "theme_1_pathway_ids", "theme_2", "theme_2_pathway_ids",
                    "theme_grouping_reason", "supporting_gene_ids_3_to_5", "mechanism",
                    "senescence_plausibility_in_carrying_classes", "strongest_alternative",
                    "citations", "short_conclusion", "appendix_theme_comparison",
                    "appendix_material_difference"]
    for column in human_fields:
        review[column] = ""
    if review_path.exists():
        saved = pd.read_csv(review_path, keep_default_na=False)
        assert saved.program_id.tolist() == review.program_id.tolist(), (
            "An existing interpretation table describes different candidates; preserve it and use a new output folder.")
        review[human_fields] = saved[human_fields]     # recompute numerical fields; preserve human judgments
    review.to_csv(review_path, index=False)
    # Table skeletons contain every computed quantity and empty human judgments.
    # Their filenames explicitly warn that they reveal the withheld benchmark.
    main = review.merge(results["senmayo", "raw"][["program_id", "NES", "q_BH", "status"]],
                        on="program_id", validate="one_to_one")
    main = main.rename(columns={"NES": "senmayo_NES", "q_BH": "senmayo_q_BH",
                                "status": "senmayo_assessability"})
    support_rows, support_labels = [], []
    for row in main.itertuples():
        chosen = [s.strip() for s in str(row.supporting_gene_ids_3_to_5).split(";") if s.strip()]
        assert not chosen or (3 <= len(chosen) <= 5 and len(set(chosen)) == len(chosen)
                              and set(chosen) <= id_set), "Supporting genes: use 3--5 unique eligible IDs separated by semicolons."
        evidence = rankings[row.program_id].set_index("gene_id").loc[chosen].reset_index()
        evidence["outside_senmayo"] = ~evidence.gene_id.isin(sen_genes)
        support_rows.append(evidence)
        support_labels.append("; ".join(f"{'+' if r.raw_weight > 0 else '-' if r.raw_weight < 0 else '0'}"
            f"{r.gene_symbol}{'*' if r.outside_senmayo else ''}" for r in evidence.itertuples()))
    main["signed_supporting_genes_star_means_outside_senmayo"] = support_labels
    pd.concat(support_rows, ignore_index=True).to_csv(
        ann / "senmayo_selected_supporting_gene_evidence_for_later.csv", index=False)
    main.to_csv(ann / "senmayo_main_evidence_table_for_later.csv", index=False)
    compact_rows = []
    for row in main.itertuples():
        interpretation = "; ".join(text for text in (row.theme_1, row.theme_2) if text)
        if row.strongest_alternative:
            interpretation += f"; alternative: {row.strongest_alternative}"
        agreement = (f"NES {row.senmayo_NES:.2f}; BH q {row.senmayo_q_BH:.3g}"
                     if row.senmayo_assessability == "tested" else row.senmayo_assessability)
        compact_rows.append([f"{row.program_id} (rank {row.rank}, {row.label})",
                             row.signed_supporting_genes_star_means_outside_senmayo,
                             interpretation.strip('; '), agreement, row.short_conclusion])
    pd.DataFrame(compact_rows, columns=["Program", "Supporting genes (* outside SenMayo)",
        "Functional interpretation", "SenMayo agreement", "Conclusion"]).to_csv(
        ann / "senmayo_compact_thesis_table_for_later.csv", index=False)
    appendix = review[["program_id", "rank", "theme_1", "theme_2", "appendix_theme_comparison",
                       "appendix_material_difference"]].merge(benchmark[
        ["program_id", "NES_raw", "q_BH_raw", "NES_specificity", "q_BH_specificity"]],
        on="program_id", validate="one_to_one")
    appendix.to_csv(ann / "senmayo_appendix_table_for_later.csv", index=False)
    top.copy().rename_axis("program_id").to_csv(ann / "selected_programs_machine_evidence.csv")
    manifest = {"fingerprint": fingerprint, "reference_sha256": hashes, "parameters": parameters,
                "gseapy_version": "1.3.1", "program_ids": [int(p) for p in top.index],
                "gene_universe_size": len(ids), "excluded_genes": int((~keep).sum()),
                "BH_families": {f"{c}_{m}": {"planned": len(t), "estimated": int(t.p_value.notna().sum()),
                                             "unavailable": int(t.p_value.isna().sum())}
                                for (c, m), t in results.items()},
                "SenMayo_results_displayed": False,
                "interpretations_are_human_judgments": True}
    (ann / "annotation_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    (ann / "READ_ME_FIRST.txt").write_text(
        "Start with the raw Reactome results, leading-gene evidence and activity figures.\n"
        "Reactome mouse pathways are largely inferred by homology from curated human pathways;\n"
        "the frozen metadata retains evidence codes. Cellular Senescence and its descendants\n"
        "are flagged for review, but enrichment tests the full eligible pathway collection.\n"
        "Complete and preserve the functional interpretation CSV before opening any senmayo_* file.\n"
        "Choose up to two themes, record grouping reasons and 3--5 signed supporting genes;\n"
        "Use semicolon-separated gene IDs in the review table. Rerun cell 2 after writing it\n"
        "to assemble the compact thesis table; cached enrichment is reused.\n"
        "Supporting-gene signs describe dictionary loadings, not observed young-versus-aged changes.\n"
        "state mechanism, plausibility in the carrying classes, strongest alternative and citations.\n"
        "Do not substitute automated enrichment for those scientific judgments.\n"
        "Then use SenMayo as a separate sanity check and annotate genes outside its published panel.\n"
        "For the appendix, compare the fixed primary themes with specificity-ranked Reactome\n"
        "before opening specificity-ranked SenMayo. Keep primary interpretations unchanged.\n"
        "The four BH families are separate; never choose the smaller q across rankings.\n"
        "Unavailable enrichment estimates retain status and missing p/q; they are not negative findings.\n"
        "BH keeps the full prespecified family, using p=1 placeholders for numerical failures only.\n"
        "NES direction describes ranking, not measured pathway activation; inspect actual weight signs.\n",
        encoding="utf-8")
    print(f"Saved molecular evidence to {ann}. SenMayo results were not displayed.")
    print("Primary Reactome overview: up to six pathways per top K program; full results are saved.")
    display(primary.groupby("program_id", sort=False).head(6)[
        ["program_id", "age_score_rank", "pathway_name", "NES", "q_BH", "universe_overlap"]])
    print(f"Scientific interpretation working table: {review_path}")
    return {"directory": ann, "review_path": review_path, "fingerprint": fingerprint,
            "primary_reactome": primary, "reactome_comparison": comparison}


annotation_outputs = annotate_top_programs(D, gene_ids, gene_symbols, top, REF, OUT)

print(f"Cell 2: {time.perf_counter() - started:.1f} seconds. Outputs: {OUT}")
